# Ejercicio 2 – Perfilamiento y preparación de T_DIGITURNOS

## Prueba Analista BI

### Objetivo
Preparar una base analítica de turnos que permita analizar tiempos de espera,
picos de demanda, cuellos de botella y utilización de usuarios receptores.

### Fuentes utilizadas
- T_DIGITURNOS.csv
- BD_AFILIADOS.csv
- AFILIADOS_A_CARGO.xlsx

### Regla de clasificación de usuarios
La identificación del cliente se cruza siguiendo este orden:

1. BD_AFILIADOS → **Afiliado**
2. Registros no encontrados → AFILIADOS_A_CARGO → **Grupo familiar**
3. Registros restantes → **No afiliado**

### Variables analíticas a construir
- tipo_vinculo
- categoría
- segmento poblacional
- pirámide1
- pirámide2
- calidad de segmentación
- día de la semana
- hora de solicitud
- tiempo de espera
- tiempo de atención
- tiempo total del servicio

### Principio de trazabilidad
Durante el proceso se documentarán:
- cantidad de registros de entrada;
- registros eliminados y motivo;
- nulos;
- duplicados;
- conversiones de tipos;
- resultados de cada cruce;
- registros sin correspondencia;
- reglas utilizadas para construir indicadores.

In [28]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. CONFIGURACIÓN DE RUTAS
# ============================================================
# El notebook se encuentra en /notebooks.
# Se toma como raíz del proyecto la carpeta inmediatamente
# superior para que las rutas sean relativas y reproducibles.
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

ruta_digiturnos = DATA_RAW / "T_DIGITURNOS.csv"
ruta_afiliados = DATA_RAW / "BD_AFILIADOS.csv"
ruta_grupo_familiar = DATA_RAW / "AFILIADOS_A_CARGO.xlsx"

rutas = {
    "T_DIGITURNOS": ruta_digiturnos,
    "BD_AFILIADOS": ruta_afiliados,
    "GRUPO_FAMILIAR": ruta_grupo_familiar
}

print("RAÍZ DEL PROYECTO:")
print(BASE_DIR)

print("\nVALIDACIÓN DE ARCHIVOS:")
for nombre, ruta in rutas.items():
    estado = "OK" if ruta.exists() else "NO ENCONTRADO"
    print(f"{nombre:<20} | {estado:<15} | {ruta}")

RAÍZ DEL PROYECTO:
c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI

VALIDACIÓN DE ARCHIVOS:
T_DIGITURNOS         | OK              | c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\raw\T_DIGITURNOS.csv
BD_AFILIADOS         | OK              | c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\raw\BD_AFILIADOS.csv
GRUPO_FAMILIAR       | OK              | c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\raw\AFILIADOS_A_CARGO.xlsx


### 1.2 Registro de trazabilidad

Se mantiene un registro de las principales transformaciones realizadas sobre las
fuentes de información. Para cada etapa se documenta la tabla intervenida, el
número de registros resultantes, el número de variables y la decisión metodológica
aplicada.

Este registro permite verificar la evolución de los datos desde su estado original
hasta la construcción de la base analítica.

In [29]:
# ============================================================
# 1.2 REGISTRO DE TRAZABILIDAD
# ============================================================

trazabilidad = []

print("Registro de trazabilidad inicializado.")

Registro de trazabilidad inicializado.


## 2.1 Diagnóstico del formato de T_DIGITURNOS

Antes de cargar la fuente se valida su estructura física para identificar
el delimitador y la codificación utilizados en el archivo.

Esta validación se realiza antes de cualquier transformación para evitar
la pérdida silenciosa de registros durante la lectura.

In [30]:
# ============================================================
# 2.1 DIAGNÓSTICO DEL FORMATO DE T_DIGITURNOS
# ============================================================

import csv

# Leer una muestra del archivo sin realizar transformaciones
with open(ruta_digiturnos, "r", encoding="utf-8-sig", errors="replace") as archivo:
    muestra = archivo.read(10000)

print("PRIMEROS 1.000 CARACTERES DEL ARCHIVO")
print("-" * 60)
print(muestra[:1000])

print("\n" + "=" * 60)
print("DETECCIÓN AUTOMÁTICA DEL DELIMITADOR")
print("=" * 60)

try:
    dialecto = csv.Sniffer().sniff(
        muestra,
        delimiters=[",", ";", "\t", "|"]
    )
    
    separador_detectado = dialecto.delimiter
    
    print(f"Separador detectado: {repr(separador_detectado)}")

except csv.Error:
    separador_detectado = None
    print("No fue posible detectar automáticamente el separador.")

PRIMEROS 1.000 CARACTERES DEL ARCHIVO
------------------------------------------------------------
Sede;UES;Clasifiacion;Usuario_receptor;Tipo_servicio;SUBSERVICIO_HOMOLOGADO;Fecha_Servico;Hora_solicitud_Servicio;Hora_llamado__Servicio;Hora_finalizado_Servicio;Identificacion_Cliente;Tipo_Oficina
Calle 26;Subsidio;Radicacion afiliaciones;39767666;Servicio;Rad afiliacion trabajadores hasta 10 formularios;20/03/2024;10:49:41;10:54:47;10:56:03;29012215;Bogota
Calle 26;Subsidio;Radicacion afiliaciones;52622358;Servicio;Rad afiliacion trabajadores hasta 10 formularios;7/09/2024;10:08:56;10:09:18;10:12:34;20000240;Bogota
Calle 26;Subsidio;Radicacion afiliaciones;52622358;Servicio;Rad afiliacion trabajadores hasta 10 formularios;7/09/2024;10:12:34;10:12:34;10:16:37;20000240;Bogota
Calle 26;Subsidio;Tarjeta multiservicios;1024494171;Servicio;Entrega tams primera vez;2/09/2024;9:22:11;9:22:11;9:28:19;20000187;Bogota
Calle 26;Subsidio;Tarjeta multiservicios;1024494171;Servicio;Entrega tams primer

In [31]:
# ============================================================
# 2.2 INSPECCIÓN DE LAS PRIMERAS LÍNEAS
# ============================================================

with open(ruta_digiturnos, "r", encoding="utf-8-sig", errors="replace") as archivo:
    for numero, linea in enumerate(archivo):
        print(f"Línea {numero + 1}: {linea[:500]}")
        
        if numero >= 4:
            break

Línea 1: Sede;UES;Clasifiacion;Usuario_receptor;Tipo_servicio;SUBSERVICIO_HOMOLOGADO;Fecha_Servico;Hora_solicitud_Servicio;Hora_llamado__Servicio;Hora_finalizado_Servicio;Identificacion_Cliente;Tipo_Oficina

Línea 2: Calle 26;Subsidio;Radicacion afiliaciones;39767666;Servicio;Rad afiliacion trabajadores hasta 10 formularios;20/03/2024;10:49:41;10:54:47;10:56:03;29012215;Bogota

Línea 3: Calle 26;Subsidio;Radicacion afiliaciones;52622358;Servicio;Rad afiliacion trabajadores hasta 10 formularios;7/09/2024;10:08:56;10:09:18;10:12:34;20000240;Bogota

Línea 4: Calle 26;Subsidio;Radicacion afiliaciones;52622358;Servicio;Rad afiliacion trabajadores hasta 10 formularios;7/09/2024;10:12:34;10:12:34;10:16:37;20000240;Bogota

Línea 5: Calle 26;Subsidio;Tarjeta multiservicios;1024494171;Servicio;Entrega tams primera vez;2/09/2024;9:22:11;9:22:11;9:28:19;20000187;Bogota



In [32]:
# ============================================================
# 2.3 INSPECCIÓN DEL REGISTRO REPORTADO POR EL PARSER
# ============================================================

lineas_revision = range(1974, 1980)

with open(ruta_digiturnos, "r", encoding="utf-8-sig", errors="replace") as archivo:
    for numero, linea in enumerate(archivo, start=1):
        if numero in lineas_revision:
            print(f"Línea {numero}:")
            print(repr(linea[:1000]))
            print("-" * 60)

Línea 1974:
'Calle 26;Subsidio;Subsidio de vivienda;52622358;Servicio;Informacion general subsidio de vivienda;27/02/2024;16:54:57;16:56:12;17:05:40;11404157;Bogota\n'
------------------------------------------------------------
Línea 1975:
'Calle 26;Subsidio;Pagos;1103714304;Servicio;Cambio de clave por olvido;19/02/2024;14:53:20;14:53:20;14:54:03;11403675;Bogota\n'
------------------------------------------------------------
Línea 1976:
'Calle 26;Subsidio;Pagos;1103714304;Servicio;Cambio de clave por olvido;19/02/2024;14:50:11;14:50:32;14:53:20;11403675;Bogota\n'
------------------------------------------------------------
Línea 1977:
'Calle 26;Educacion;Educacion;1033797428;Comercial;Informacion, inscripcion cet tyt y tl;27/06/2024;16:18:44;16:25:05;16:31:08;11372219;Bogota\n'
------------------------------------------------------------
Línea 1978:
'Calle 26;Subsidio;Radicacion afiliaciones;1024480700;Servicio;Rad afiliacion trabajadores hasta 10 formularios;27/06/2024;16:56:23;16:5

### Hallazgo de ingesta

El archivo `T_DIGITURNOS.csv` utiliza punto y coma (`;`) como delimitador.

La carga inicial con el separador predeterminado de `pandas.read_csv()` produjo un
`ParserError` en la línea 1977. La inspección del archivo permitió establecer que
el problema no correspondía a una fila corrupta, sino a una coma incluida dentro
del texto de un subservicio:

`Informacion, inscripcion cet tyt y tl`

Al utilizar `;` como delimitador, esta coma se conserva correctamente como parte
del contenido del campo.

**Decisión:** cargar la fuente con `sep=";"` y conservar todos los registros,
sin utilizar `on_bad_lines="skip"`.

## 2. Carga de fuentes

Se cargan las tres fuentes originales sin modificar los archivos de entrada.

Los archivos ubicados en `data/raw` se consideran fuentes inmutables.
Todas las transformaciones posteriores se realizan en memoria y los resultados
procesados se almacenarán en `data/processed`.

Esto permite conservar los datos originales y reproducir completamente el proceso.

In [33]:
# ============================================================
# 2.4 CARGA CONTROLADA DE FUENTES
# ============================================================
# T_DIGITURNOS utiliza punto y coma (;) como delimitador.
# No se descartan registros durante la lectura.
# ============================================================

archivos_faltantes = [
    nombre
    for nombre, ruta in rutas.items()
    if not ruta.exists()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos: "
        + ", ".join(archivos_faltantes)
    )

digiturnos = pd.read_csv(
    ruta_digiturnos,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

afiliados = pd.read_csv(
    ruta_afiliados,
    low_memory=False
)

grupo_familiar = pd.read_excel(
    ruta_grupo_familiar
)

print("CARGA COMPLETADA")
print("=" * 60)
print(
    f"T_DIGITURNOS:   "
    f"{digiturnos.shape[0]:,} filas x "
    f"{digiturnos.shape[1]} columnas"
)
print(
    f"BD_AFILIADOS:   "
    f"{afiliados.shape[0]:,} filas x "
    f"{afiliados.shape[1]} columnas"
)
print(
    f"GRUPO_FAMILIAR: "
    f"{grupo_familiar.shape[0]:,} filas x "
    f"{grupo_familiar.shape[1]} columnas"
)

CARGA COMPLETADA
T_DIGITURNOS:   1,016,267 filas x 12 columnas
BD_AFILIADOS:   448,951 filas x 1 columnas
GRUPO_FAMILIAR: 39,733 filas x 2 columnas


In [34]:
# ============================================================
# REGISTRO DE TRAZABILIDAD - CARGA ORIGINAL
# ============================================================

trazabilidad.append({
    "etapa": "Carga de fuente",
    "tabla": "T_DIGITURNOS",
    "registros": len(digiturnos),
    "columnas": digiturnos.shape[1],
    "observacion": (
        "Fuente cargada con separador ';'. "
        "No se descartaron registros durante la ingesta."
    )
})

trazabilidad.append({
    "etapa": "Carga de fuente",
    "tabla": "BD_AFILIADOS",
    "registros": len(afiliados),
    "columnas": afiliados.shape[1],
    "observacion": "Fuente cargada sin transformaciones."
})

trazabilidad.append({
    "etapa": "Carga de fuente",
    "tabla": "GRUPO_FAMILIAR",
    "registros": len(grupo_familiar),
    "columnas": grupo_familiar.shape[1],
    "observacion": "Fuente cargada sin transformaciones."
})

display(pd.DataFrame(trazabilidad))

,etapa,tabla,registros,columnas,observacion
0,Carga de fuente,T_DIGITURNOS,1016267,12,Fuente cargada con separador ';'. No se descar...
1,Carga de fuente,BD_AFILIADOS,448951,1,Fuente cargada sin transformaciones.
2,Carga de fuente,GRUPO_FAMILIAR,39733,2,Fuente cargada sin transformaciones.


In [35]:
# ============================================================
# 2.5 VALIDACIÓN ESTRUCTURAL POST-CARGA
# ============================================================

print("COLUMNAS T_DIGITURNOS")
print("=" * 60)

for i, columna in enumerate(digiturnos.columns, start=1):
    print(f"{i:02d}. {columna}")

print("\nNúmero de columnas:", digiturnos.shape[1])

display(digiturnos.head())

COLUMNAS T_DIGITURNOS
01. Sede
02. UES
03. Clasifiacion
04. Usuario_receptor
05. Tipo_servicio
06. SUBSERVICIO_HOMOLOGADO
07. Fecha_Servico
08. Hora_solicitud_Servicio
09. Hora_llamado__Servicio
10. Hora_finalizado_Servicio
11. Identificacion_Cliente
12. Tipo_Oficina

Número de columnas: 12


,Sede,UES,Clasifiacion,Usuario_receptor,Tipo_servicio,SUBSERVICIO_HOMOLOGADO,Fecha_Servico,Hora_solicitud_Servicio,Hora_llamado__Servicio,Hora_finalizado_Servicio,Identificacion_Cliente,Tipo_Oficina
0,Calle 26,Subsidio,Radicacion afiliaciones,39767666,Servicio,Rad afiliacion trabajadores hasta 10 formularios,20/03/2024,10:49:41,10:54:47,10:56:03,29012215.0,Bogota
1,Calle 26,Subsidio,Radicacion afiliaciones,52622358,Servicio,Rad afiliacion trabajadores hasta 10 formularios,7/09/2024,10:08:56,10:09:18,10:12:34,20000240.0,Bogota
2,Calle 26,Subsidio,Radicacion afiliaciones,52622358,Servicio,Rad afiliacion trabajadores hasta 10 formularios,7/09/2024,10:12:34,10:12:34,10:16:37,20000240.0,Bogota
3,Calle 26,Subsidio,Tarjeta multiservicios,1024494171,Servicio,Entrega tams primera vez,2/09/2024,9:22:11,9:22:11,9:28:19,20000187.0,Bogota
4,Calle 26,Subsidio,Tarjeta multiservicios,1024494171,Servicio,Entrega tams primera vez,2/09/2024,9:21:12,9:21:12,9:22:11,20000187.0,Bogota


In [36]:
# ============================================================
# 2.5 VALIDACIÓN ESTRUCTURAL POST-CARGA
# ============================================================

print("COLUMNAS T_DIGITURNOS")
print("=" * 60)

for i, columna in enumerate(digiturnos.columns, start=1):
    print(f"{i:02d}. {columna}")

print("\nNúmero de columnas:", digiturnos.shape[1])

display(digiturnos.head())

COLUMNAS T_DIGITURNOS
01. Sede
02. UES
03. Clasifiacion
04. Usuario_receptor
05. Tipo_servicio
06. SUBSERVICIO_HOMOLOGADO
07. Fecha_Servico
08. Hora_solicitud_Servicio
09. Hora_llamado__Servicio
10. Hora_finalizado_Servicio
11. Identificacion_Cliente
12. Tipo_Oficina

Número de columnas: 12


,Sede,UES,Clasifiacion,Usuario_receptor,Tipo_servicio,SUBSERVICIO_HOMOLOGADO,Fecha_Servico,Hora_solicitud_Servicio,Hora_llamado__Servicio,Hora_finalizado_Servicio,Identificacion_Cliente,Tipo_Oficina
0,Calle 26,Subsidio,Radicacion afiliaciones,39767666,Servicio,Rad afiliacion trabajadores hasta 10 formularios,20/03/2024,10:49:41,10:54:47,10:56:03,29012215.0,Bogota
1,Calle 26,Subsidio,Radicacion afiliaciones,52622358,Servicio,Rad afiliacion trabajadores hasta 10 formularios,7/09/2024,10:08:56,10:09:18,10:12:34,20000240.0,Bogota
2,Calle 26,Subsidio,Radicacion afiliaciones,52622358,Servicio,Rad afiliacion trabajadores hasta 10 formularios,7/09/2024,10:12:34,10:12:34,10:16:37,20000240.0,Bogota
3,Calle 26,Subsidio,Tarjeta multiservicios,1024494171,Servicio,Entrega tams primera vez,2/09/2024,9:22:11,9:22:11,9:28:19,20000187.0,Bogota
4,Calle 26,Subsidio,Tarjeta multiservicios,1024494171,Servicio,Entrega tams primera vez,2/09/2024,9:21:12,9:21:12,9:22:11,20000187.0,Bogota


In [37]:
# ============================================================
# 3. PERFILAMIENTO INICIAL
# 3.1 Registros completamente vacíos
# ============================================================

filas_iniciales = len(digiturnos)
filas_vacias = digiturnos.isna().all(axis=1).sum()

print(f"Filas cargadas:              {filas_iniciales:,}")
print(f"Filas completamente vacías:  {filas_vacias:,}")
print(
    f"Porcentaje de filas vacías:  "
    f"{filas_vacias / filas_iniciales:.2%}"
)

Filas cargadas:              1,016,267
Filas completamente vacías:  133,187
Porcentaje de filas vacías:  13.11%


In [38]:
# ============================================================
# 3.2 ELIMINACIÓN CONTROLADA DE FILAS VACÍAS
# ============================================================

digiturnos = digiturnos.dropna(how="all").copy()

filas_utiles = len(digiturnos)

print(f"Filas antes:       {filas_iniciales:,}")
print(f"Filas eliminadas:  {filas_vacias:,}")
print(f"Filas útiles:      {filas_utiles:,}")

Filas antes:       1,016,267
Filas eliminadas:  133,187
Filas útiles:      883,080


In [39]:
# ============================================================
# 3.2.1 CONTROL DE LA LIMPIEZA DE FILAS VACÍAS
# ============================================================

print("CONTROL POST-LIMPIEZA")
print("=" * 60)

print(f"Registros originales:          {filas_iniciales:>12,}")
print(f"Filas completamente vacías:    {filas_vacias:>12,}")
print(f"Registros analíticos:          {len(digiturnos):>12,}")
print("-" * 60)
print(
    "Validación aritmética:        ",
    filas_iniciales == filas_vacias + len(digiturnos)
)

filas_vacias_post = digiturnos.isna().all(axis=1).sum()
print(f"Filas vacías después limpieza: {filas_vacias_post:>12,}")

CONTROL POST-LIMPIEZA
Registros originales:             1,016,267
Filas completamente vacías:         133,187
Registros analíticos:               883,080
------------------------------------------------------------
Validación aritmética:         True
Filas vacías después limpieza:            0


In [40]:
trazabilidad.append({
    "etapa": "Limpieza inicial",
    "tabla": "T_DIGITURNOS",
    "registros": filas_utiles,
    "columnas": digiturnos.shape[1],
    "observacion": (
        f"Se eliminaron {filas_vacias:,} filas "
        "completamente vacías."
    )
})

display(pd.DataFrame(trazabilidad))

,etapa,tabla,registros,columnas,observacion
0,Carga de fuente,T_DIGITURNOS,1016267,12,Fuente cargada con separador ';'. No se descar...
1,Carga de fuente,BD_AFILIADOS,448951,1,Fuente cargada sin transformaciones.
2,Carga de fuente,GRUPO_FAMILIAR,39733,2,Fuente cargada sin transformaciones.
3,Limpieza inicial,T_DIGITURNOS,883080,12,"Se eliminaron 133,187 filas completamente vacías."


In [41]:
# ============================================================
# 3.3 PERFIL DE CALIDAD POR VARIABLE
# ============================================================

perfil_calidad = pd.DataFrame({
    "tipo_dato": digiturnos.dtypes.astype(str),
    "registros": len(digiturnos),
    "nulos": digiturnos.isna().sum(),
    "pct_nulos": (digiturnos.isna().mean() * 100).round(2),
    "distintos": digiturnos.nunique(dropna=True)
})

perfil_calidad = perfil_calidad.sort_values(
    "pct_nulos",
    ascending=False
)

display(perfil_calidad)

,tipo_dato,registros,nulos,pct_nulos,distintos
Sede,str,883080,0,0.0,13
UES,str,883080,0,0.0,6
Clasifiacion,str,883080,0,0.0,12
Usuario_receptor,str,883080,0,0.0,122
Tipo_servicio,str,883080,0,0.0,2
SUBSERVICIO_HOMOLOGADO,str,883080,0,0.0,140
Fecha_Servico,str,883080,0,0.0,212
Hora_solicitud_Servicio,str,883080,0,0.0,44170
Hora_llamado__Servicio,str,883080,0,0.0,44352
Hora_finalizado_Servicio,str,883080,0,0.0,44401


In [42]:
# ============================================================
# 3.4 DUPLICADOS EXACTOS
# ============================================================

duplicados_exactos = digiturnos.duplicated().sum()
pct_duplicados = duplicados_exactos / len(digiturnos) * 100

print("CONTROL DE DUPLICADOS")
print("=" * 60)
print(f"Registros analizados:     {len(digiturnos):,}")
print(f"Duplicados exactos:       {duplicados_exactos:,}")
print(f"Porcentaje duplicados:    {pct_duplicados:.2f}%")

CONTROL DE DUPLICADOS
Registros analizados:     883,080
Duplicados exactos:       0
Porcentaje duplicados:    0.00%


### 3.5 Diagnóstico de variables críticas

Antes de construir los indicadores de tiempos de atención y realizar el cruce con
la población afiliada, se valida la calidad de las variables que intervienen
directamente en el análisis.

Se revisan especialmente la identificación del cliente, la fecha del servicio,
las horas de solicitud, llamado y finalización, y el usuario receptor. El objetivo
es identificar valores no válidos o inconsistencias antes de realizar
transformaciones que puedan alterar los registros originales.

In [43]:
# ============================================================
# 3.5 DIAGNÓSTICO DE VARIABLES CRÍTICAS
# ============================================================

columnas_criticas = [
    "Identificacion_Cliente",
    "Fecha_Servico",
    "Hora_solicitud_Servicio",
    "Hora_llamado__Servicio",
    "Hora_finalizado_Servicio",
    "Usuario_receptor"
]

print("DIAGNÓSTICO DE VARIABLES CRÍTICAS")
print("=" * 70)

for columna in columnas_criticas:
    print(f"\n{columna}")
    print("-" * 70)
    print(f"Tipo de dato:       {digiturnos[columna].dtype}")
    print(f"Registros:          {len(digiturnos):,}")
    print(f"Nulos:              {digiturnos[columna].isna().sum():,}")
    print(f"Valores distintos:  {digiturnos[columna].nunique(dropna=True):,}")

DIAGNÓSTICO DE VARIABLES CRÍTICAS

Identificacion_Cliente
----------------------------------------------------------------------
Tipo de dato:       float64
Registros:          883,080
Nulos:              0
Valores distintos:  60,335

Fecha_Servico
----------------------------------------------------------------------
Tipo de dato:       str
Registros:          883,080
Nulos:              0
Valores distintos:  212

Hora_solicitud_Servicio
----------------------------------------------------------------------
Tipo de dato:       str
Registros:          883,080
Nulos:              0
Valores distintos:  44,170

Hora_llamado__Servicio
----------------------------------------------------------------------
Tipo de dato:       str
Registros:          883,080
Nulos:              0
Valores distintos:  44,352

Hora_finalizado_Servicio
----------------------------------------------------------------------
Tipo de dato:       str
Registros:          883,080
Nulos:              0
Valores distintos:

In [44]:
# ============================================================
# 3.6 DIAGNÓSTICO DEL IDENTIFICADOR DEL CLIENTE
# ============================================================

id_cliente = pd.to_numeric(
    digiturnos["Identificacion_Cliente"],
    errors="coerce"
)

ids_no_numericos = id_cliente.isna().sum()

ids_con_decimales = (
    id_cliente.notna()
    & (id_cliente % 1 != 0)
).sum()

print("CONTROL DEL IDENTIFICADOR DEL CLIENTE")
print("=" * 70)
print(f"Registros analizados:        {len(digiturnos):,}")
print(f"Valores no numéricos:        {ids_no_numericos:,}")
print(f"Valores con parte decimal:   {ids_con_decimales:,}")
print(f"Identificaciones distintas:  {id_cliente.nunique():,}")

CONTROL DEL IDENTIFICADOR DEL CLIENTE
Registros analizados:        883,080
Valores no numéricos:        0
Valores con parte decimal:   0
Identificaciones distintas:  60,335


In [45]:
# ============================================================
# 3.7 VALIDACIÓN DE FECHAS Y HORAS
# ============================================================

fecha_test = pd.to_datetime(
    digiturnos["Fecha_Servico"],
    format="%d/%m/%Y",
    errors="coerce"
)

hora_solicitud_test = pd.to_timedelta(
    digiturnos["Hora_solicitud_Servicio"],
    errors="coerce"
)

hora_llamado_test = pd.to_timedelta(
    digiturnos["Hora_llamado__Servicio"],
    errors="coerce"
)

hora_finalizado_test = pd.to_timedelta(
    digiturnos["Hora_finalizado_Servicio"],
    errors="coerce"
)

validacion_temporal = pd.DataFrame({
    "variable": [
        "Fecha_Servico",
        "Hora_solicitud_Servicio",
        "Hora_llamado__Servicio",
        "Hora_finalizado_Servicio"
    ],
    "registros": [len(digiturnos)] * 4,
    "no_convertibles": [
        fecha_test.isna().sum(),
        hora_solicitud_test.isna().sum(),
        hora_llamado_test.isna().sum(),
        hora_finalizado_test.isna().sum()
    ]
})

validacion_temporal["pct_no_convertibles"] = (
    validacion_temporal["no_convertibles"]
    / validacion_temporal["registros"]
    * 100
).round(4)

display(validacion_temporal)

,variable,registros,no_convertibles,pct_no_convertibles
0,Fecha_Servico,883080,0,0.0
1,Hora_solicitud_Servicio,883080,0,0.0
2,Hora_llamado__Servicio,883080,0,0.0
3,Hora_finalizado_Servicio,883080,0,0.0


### 3.8 Estandarización de variables críticas

Las validaciones previas evidenciaron que la identificación del cliente no contiene
valores no numéricos ni componentes decimales reales. Asimismo, la fecha del
servicio y las tres variables horarias presentan una conversión válida para la
totalidad de los registros analíticos.

A partir de estos resultados se estandariza el identificador como texto, con el
fin de utilizarlo posteriormente como llave de integración, y se construyen
marcas de tiempo completas para la solicitud, el llamado y la finalización del
servicio.

Las variables originales se conservan para garantizar la trazabilidad del
proceso.

In [46]:
# ============================================================
# 3.8 ESTANDARIZACIÓN DE VARIABLES CRÍTICAS
# ============================================================

# ------------------------------------------------------------
# 1. Identificación del cliente
# ------------------------------------------------------------

digiturnos["id_cliente"] = (
    pd.to_numeric(
        digiturnos["Identificacion_Cliente"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


# ------------------------------------------------------------
# 2. Fecha del servicio
# ------------------------------------------------------------

digiturnos["fecha_servicio"] = pd.to_datetime(
    digiturnos["Fecha_Servico"],
    format="%d/%m/%Y",
    errors="coerce"
)


# ------------------------------------------------------------
# 3. Construcción de timestamps
# ------------------------------------------------------------

digiturnos["ts_solicitud"] = (
    digiturnos["fecha_servicio"]
    + pd.to_timedelta(
        digiturnos["Hora_solicitud_Servicio"],
        errors="coerce"
    )
)

digiturnos["ts_llamado"] = (
    digiturnos["fecha_servicio"]
    + pd.to_timedelta(
        digiturnos["Hora_llamado__Servicio"],
        errors="coerce"
    )
)

digiturnos["ts_finalizacion"] = (
    digiturnos["fecha_servicio"]
    + pd.to_timedelta(
        digiturnos["Hora_finalizado_Servicio"],
        errors="coerce"
    )
)


# ------------------------------------------------------------
# 4. Control de transformación
# ------------------------------------------------------------

print("ESTANDARIZACIÓN DE VARIABLES CRÍTICAS")
print("=" * 70)

print(f"Registros procesados:           {len(digiturnos):,}")
print(f"ID cliente nulos:               {digiturnos['id_cliente'].isna().sum():,}")
print(f"Fecha servicio nula:            {digiturnos['fecha_servicio'].isna().sum():,}")
print(f"Timestamp solicitud nulo:       {digiturnos['ts_solicitud'].isna().sum():,}")
print(f"Timestamp llamado nulo:         {digiturnos['ts_llamado'].isna().sum():,}")
print(f"Timestamp finalización nulo:    {digiturnos['ts_finalizacion'].isna().sum():,}")

ESTANDARIZACIÓN DE VARIABLES CRÍTICAS
Registros procesados:           883,080
ID cliente nulos:               0
Fecha servicio nula:            0
Timestamp solicitud nulo:       0
Timestamp llamado nulo:         0
Timestamp finalización nulo:    0


### 3.9 Construcción y validación de tiempos operativos

A partir de las marcas de tiempo estandarizadas se calculan dos componentes del
proceso de atención:

- **Tiempo de espera:** minutos transcurridos entre la solicitud del turno y el llamado.
- **Tiempo de atención:** minutos transcurridos entre el llamado y la finalización del servicio.

Antes de utilizar estos indicadores se verifica su coherencia temporal. Los
valores negativos se identifican como casos potencialmente inconsistentes y no
se eliminan automáticamente, con el propósito de conservar evidencia para el
control de calidad.

In [47]:
# ============================================================
# 3.9 CONSTRUCCIÓN Y CONTROL DE TIEMPOS OPERATIVOS
# ============================================================

digiturnos["tiempo_espera_min"] = (
    (
        digiturnos["ts_llamado"]
        - digiturnos["ts_solicitud"]
    )
    .dt.total_seconds()
    / 60
)

digiturnos["tiempo_atencion_min"] = (
    (
        digiturnos["ts_finalizacion"]
        - digiturnos["ts_llamado"]
    )
    .dt.total_seconds()
    / 60
)

digiturnos["tiempo_total_min"] = (
    (
        digiturnos["ts_finalizacion"]
        - digiturnos["ts_solicitud"]
    )
    .dt.total_seconds()
    / 60
)


# ------------------------------------------------------------
# Controles de coherencia temporal
# ------------------------------------------------------------

esperas_negativas = (digiturnos["tiempo_espera_min"] < 0).sum()
atenciones_negativas = (digiturnos["tiempo_atencion_min"] < 0).sum()
totales_negativos = (digiturnos["tiempo_total_min"] < 0).sum()

esperas_cero = (digiturnos["tiempo_espera_min"] == 0).sum()
atenciones_cero = (digiturnos["tiempo_atencion_min"] == 0).sum()


print("CONTROL DE COHERENCIA TEMPORAL")
print("=" * 70)

print(f"Registros analizados:             {len(digiturnos):,}")
print()
print(f"Tiempos de espera negativos:      {esperas_negativas:,}")
print(f"Tiempos de atención negativos:    {atenciones_negativas:,}")
print(f"Tiempos totales negativos:        {totales_negativos:,}")
print()
print(f"Tiempos de espera = 0:            {esperas_cero:,}")
print(f"Tiempos de atención = 0:          {atenciones_cero:,}")

CONTROL DE COHERENCIA TEMPORAL
Registros analizados:             883,080

Tiempos de espera negativos:      2
Tiempos de atención negativos:    2
Tiempos totales negativos:        4

Tiempos de espera = 0:            347,996
Tiempos de atención = 0:          2


In [48]:
# ============================================================
# 3.10 DISTRIBUCIÓN PRELIMINAR DE TIEMPOS
# ============================================================

variables_tiempo = [
    "tiempo_espera_min",
    "tiempo_atencion_min",
    "tiempo_total_min"
]

resumen_tiempos = (
    digiturnos[variables_tiempo]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

display(
    resumen_tiempos[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "90%",
            "95%",
            "99%",
            "max"
        ]
    ].round(2)
)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
tiempo_espera_min,883080.0,6.04,10.21,-786.90,0.00,0.78,8.45,18.92,26.98,45.82,200.62
tiempo_atencion_min,883080.0,5.14,4.40,-578.18,2.33,4.15,6.77,10.13,12.90,20.52,171.25
tiempo_total_min,883080.0,11.18,11.43,-786.90,3.67,7.35,14.98,25.77,34.08,53.29,200.70


### 3.11 Tratamiento de inconsistencias temporales

El control de coherencia identificó un número marginal de registros con secuencias
temporales inconsistentes. Estos casos se conservan en la base para mantener la
trazabilidad de la fuente, pero se clasifican mediante una bandera de calidad.

Para el cálculo posterior de indicadores de tiempos se utilizarán únicamente
registros con secuencia temporal válida. Los tiempos iguales a cero se mantienen,
dado que pueden representar atenciones inmediatas y no constituyen por sí mismos
una inconsistencia.

Los valores extremos positivos tampoco se eliminan en esta etapa. Su comportamiento
será analizado posteriormente por sede y tipo de servicio, dado que pueden
representar eventos operativos reales y ser relevantes para la identificación
de cuellos de botella.

In [49]:
# ============================================================
# 3.11 TRATAMIENTO DE INCONSISTENCIAS TEMPORALES
# ============================================================

digiturnos["flag_tiempo_valido"] = (
    (digiturnos["tiempo_espera_min"] >= 0)
    & (digiturnos["tiempo_atencion_min"] >= 0)
    & (digiturnos["tiempo_total_min"] >= 0)
)

digiturnos["estado_calidad_tiempo"] = (
    digiturnos["flag_tiempo_valido"]
    .map({
        True: "Válido",
        False: "Inconsistente"
    })
)

control_calidad_tiempo = (
    digiturnos["estado_calidad_tiempo"]
    .value_counts(dropna=False)
    .rename_axis("estado")
    .reset_index(name="registros")
)

control_calidad_tiempo["porcentaje"] = (
    control_calidad_tiempo["registros"]
    / len(digiturnos)
    * 100
).round(4)

display(control_calidad_tiempo)

,estado,registros,porcentaje
0,Válido,883076,99.9995
1,Inconsistente,4,0.0005


In [51]:
# ============================================================
# 3.12.1 VERIFICACIÓN DE NOMBRES DE COLUMNAS
# ============================================================

print("COLUMNAS ACTUALES DE T_DIGITURNOS")
print("=" * 70)

for i, columna in enumerate(digiturnos.columns, start=1):
    print(f"{i:02d}. {repr(columna)}")

COLUMNAS ACTUALES DE T_DIGITURNOS
01. 'Sede'
02. 'UES'
03. 'Clasifiacion'
04. 'Usuario_receptor'
05. 'Tipo_servicio'
06. 'SUBSERVICIO_HOMOLOGADO'
07. 'Fecha_Servico'
08. 'Hora_solicitud_Servicio'
09. 'Hora_llamado__Servicio'
10. 'Hora_finalizado_Servicio'
11. 'Identificacion_Cliente'
12. 'Tipo_Oficina'
13. 'id_cliente'
14. 'fecha_servicio'
15. 'ts_solicitud'
16. 'ts_llamado'
17. 'ts_finalizacion'
18. 'tiempo_espera_min'
19. 'tiempo_atencion_min'
20. 'tiempo_total_min'
21. 'flag_tiempo_valido'
22. 'estado_calidad_tiempo'


In [54]:
# ============================================================
# 3.12 AUDITORÍA DE INCONSISTENCIAS TEMPORALES
# ============================================================

columnas_deseadas = [
    "Sede",
    "UES",
    "Clasificacion",
    "Usuario_receptor",
    "Tipo_servicio",
    "SUBSERVICIO_HOMOLOGADO",
    "fecha_servicio",
    "ts_solicitud",
    "ts_llamado",
    "ts_finalizacion",
    "tiempo_espera_min",
    "tiempo_atencion_min",
    "tiempo_total_min"
]

# Seleccionar únicamente columnas que realmente existen
columnas_auditoria = [
    c for c in columnas_deseadas
    if c in digiturnos.columns
]

columnas_faltantes = [
    c for c in columnas_deseadas
    if c not in digiturnos.columns
]

# Primero filtrar filas
casos_inconsistentes = (
    digiturnos.loc[
        ~digiturnos["flag_tiempo_valido"]
    ]
    .copy()
)

# Después seleccionar/reordenar columnas de forma segura
casos_inconsistentes = (
    casos_inconsistentes
    .reindex(columns=columnas_auditoria)
    .sort_values(
        "tiempo_total_min",
        ascending=True
    )
)

print("AUDITORÍA DE INCONSISTENCIAS TEMPORALES")
print("=" * 70)

print(f"Registros analizados:       {len(digiturnos):,}")
print(f"Casos inconsistentes:       {len(casos_inconsistentes):,}")

print(
    f"Porcentaje inconsistente:   "
    f"{len(casos_inconsistentes) / len(digiturnos) * 100:.4f}%"
)

print()

if columnas_faltantes:
    print("Columnas no encontradas:")
    print(columnas_faltantes)
else:
    print("Todas las columnas de auditoría fueron encontradas.")

print()

display(casos_inconsistentes)

AUDITORÍA DE INCONSISTENCIAS TEMPORALES
Registros analizados:       883,080
Casos inconsistentes:       4
Porcentaje inconsistente:   0.0005%

Columnas no encontradas:
['Clasificacion']



,Sede,UES,Usuario_receptor,Tipo_servicio,SUBSERVICIO_HOMOLOGADO,fecha_servicio,ts_solicitud,ts_llamado,ts_finalizacion,tiempo_espera_min,tiempo_atencion_min,tiempo_total_min
500217,Soacha,Hoteleria y turismo,Sin Usuario,Comercial,Radicacion reserva,2024-04-20,2024-04-20 13:06:54,2024-04-20 00:00:00,2024-04-20 00:00:00,-786.900000,0.000000,-786.900000
534463,Soacha,Credito,1024546447,Comercial,Radicacion cupo credito,2024-04-06,2024-04-06 12:26:40,2024-04-06 00:00:00,2024-04-06 00:00:00,-746.666667,0.000000,-746.666667
449138,Santa librada,Subsidio,1019125029,Servicio,Validacion afiliacion trabajador y/o beneficia...,2024-06-13,2024-06-13 16:59:03,2024-06-13 17:18:20,2024-06-13 07:40:09,19.283333,-578.183333,-558.900000
306392,Funza,Subsidio,1003822265,Servicio,Informacion subsidios de arrendamiento,2024-06-11,2024-06-11 17:55:50,2024-06-11 18:08:08,2024-06-11 08:58:29,12.300000,-549.650000,-537.350000


In [55]:
# ============================================================
# 3.13 CONSTRUCCIÓN DE POBLACIÓN VÁLIDA PARA TIEMPOS
# ============================================================

digiturnos_tiempos = (
    digiturnos.loc[
        digiturnos["flag_tiempo_valido"]
    ]
    .copy()
)

registros_excluidos = (
    len(digiturnos)
    - len(digiturnos_tiempos)
)

print("POBLACIÓN ANALÍTICA PARA INDICADORES TEMPORALES")
print("=" * 70)

print(f"Registros base:                  {len(digiturnos):>12,}")
print(f"Registros temporalmente válidos: {len(digiturnos_tiempos):>12,}")
print(f"Registros excluidos:             {registros_excluidos:>12,}")

print("-" * 70)

print(
    "Validación de reconciliación: ",
    len(digiturnos)
    == len(digiturnos_tiempos)
    + registros_excluidos
)

POBLACIÓN ANALÍTICA PARA INDICADORES TEMPORALES
Registros base:                       883,080
Registros temporalmente válidos:      883,076
Registros excluidos:                        4
----------------------------------------------------------------------
Validación de reconciliación:  True


In [56]:
trazabilidad.append({
    "etapa": "Validación temporal",
    "tabla": "T_DIGITURNOS",
    "registros": len(digiturnos_tiempos),
    "columnas": digiturnos_tiempos.shape[1],
    "observacion": (
        f"Se identificaron {registros_excluidos:,} registros "
        "con secuencia temporal inconsistente. Se conservan en "
        "la base original y se excluyen únicamente del cálculo "
        "de indicadores temporales."
    )
})

display(pd.DataFrame(trazabilidad))

,etapa,tabla,registros,columnas,observacion
0,Carga de fuente,T_DIGITURNOS,1016267,12,Fuente cargada con separador ';'. No se descar...
1,Carga de fuente,BD_AFILIADOS,448951,1,Fuente cargada sin transformaciones.
2,Carga de fuente,GRUPO_FAMILIAR,39733,2,Fuente cargada sin transformaciones.
3,Limpieza inicial,T_DIGITURNOS,883080,12,"Se eliminaron 133,187 filas completamente vacías."
4,Validación temporal,T_DIGITURNOS,883076,22,Se identificaron 4 registros con secuencia tem...


## 4. Clasificación de clientes según afiliación

La clasificación de los clientes se desarrolla mediante una estrategia jerárquica
de identificación. Primero se busca correspondencia directa con la base de
afiliados; posteriormente, los clientes sin coincidencia se contrastan con la
base de personas a cargo y, finalmente, aquellos que no presentan correspondencia
en ninguna de las dos fuentes se clasifican como no afiliados.

Antes de realizar los cruces se valida la estructura, calidad y unicidad de las
llaves de identificación utilizadas en cada fuente, con el propósito de evitar
duplicaciones o clasificaciones incorrectas durante las uniones.

In [57]:
# ============================================================
# 4.1 DIAGNÓSTICO DE LLAVES PARA CLASIFICACIÓN DE AFILIACIÓN
# ============================================================

print("DIAGNÓSTICO DE FUENTES PARA CLASIFICACIÓN")
print("=" * 70)

print("\nT_DIGITURNOS")
print("-" * 70)
print(f"Registros: {len(digiturnos):,}")
print(f"Columnas:  {digiturnos.shape[1]}")
print(f"ID clientes distintos: {digiturnos['id_cliente'].nunique():,}")

print("\nBD_AFILIADOS")
print("-" * 70)
print(f"Registros: {len(afiliados):,}")
print(f"Columnas:  {afiliados.shape[1]}")
print("Nombres de columnas:")
for i, col in enumerate(afiliados.columns, 1):
    print(f"{i:02d}. {repr(col)}")

print("\nGRUPO_FAMILIAR")
print("-" * 70)
print(f"Registros: {len(grupo_familiar):,}")
print(f"Columnas:  {grupo_familiar.shape[1]}")
print("Nombres de columnas:")
for i, col in enumerate(grupo_familiar.columns, 1):
    print(f"{i:02d}. {repr(col)}")

DIAGNÓSTICO DE FUENTES PARA CLASIFICACIÓN

T_DIGITURNOS
----------------------------------------------------------------------
Registros: 883,080
Columnas:  22
ID clientes distintos: 60,335

BD_AFILIADOS
----------------------------------------------------------------------
Registros: 448,951
Columnas:  1
Nombres de columnas:
01. ';id_persona_Modificado;NumIdPersona_Modificado;Nombre;Categoria;Segmento_poblacional;Piramide1;Piramide2;RazonSocial;Genero'

GRUPO_FAMILIAR
----------------------------------------------------------------------
Registros: 39,733
Columnas:  2
Nombres de columnas:
01. 'NUM_IDENTIFICACION_AFILIADO'
02. 'NUM_IDENTIFICACION_PERSONA_A_CARGO'


In [58]:
# ============================================================
# 4.2 MUESTRA CONTROLADA DE LAS FUENTES
# ============================================================

print("MUESTRA BD_AFILIADOS")
display(afiliados.head(10))

print("\nMUESTRA GRUPO_FAMILIAR")
display(grupo_familiar.head(10))

print("\nMUESTRA IDENTIFICADORES T_DIGITURNOS")
display(
    digiturnos[
        ["Identificacion_Cliente", "id_cliente"]
    ].head(10)
)

MUESTRA BD_AFILIADOS


,;id_persona_Modificado;NumIdPersona_Modificado;Nombre;Categoria;Segmento_poblacional;Piramide1;Piramide2;RazonSocial;Genero
0,0;CC10818289;10818289;ORLANDO JOSEALGUERO SOSA...
1,1;CC10224342;10224342;CRISTIAN LEONARDOCORTES ...
2,2;CC10814050;10814050;LAURA NIKOLERAMIREZ LOZA...
3,3;CC31064;31064;JOSE EDILBERTOBOHORQUEZ BEJARA...
4,4;CC10957942;10957942;JOSEFINA ANGARITA PINZON...
5,5;CC750837;750837;RAMON ANTONIOGIRALDO OSORIO;...
6,6;CC463793;463793;CLAUDIA MILENAGONZALEZ LEON;...
7,7;CC760418;760418;ERMINSOL SUAREZ QUINTERO;A;B...
8,8;CC170903;170903;GERARDINO HENRYCASTELLANOS D...
9,9;CC10223482;10223482;JOHN ALEXANDERRODRIGUEZ ...



MUESTRA GRUPO_FAMILIAR


,NUM_IDENTIFICACION_AFILIADO,NUM_IDENTIFICACION_PERSONA_A_CARGO
0,1775,10036404
1,3824,96022201
2,4133,99022409
3,3084,10003904
4,3859,10007324
5,4133,11411220
6,4224,10039076
7,3624,93083126
8,3282,98111927
9,4581,98110308



MUESTRA IDENTIFICADORES T_DIGITURNOS


,Identificacion_Cliente,id_cliente
0,29012215.0,29012215
1,20000240.0,20000240
2,20000240.0,20000240
3,20000187.0,20000187
4,20000187.0,20000187
5,20000187.0,20000187
6,20000187.0,20000187
7,20000187.0,20000187
8,20000187.0,20000187
9,20000187.0,20000187


### 4.3 Estandarización de las fuentes de afiliación

La inspección de `BD_AFILIADOS` evidenció que el archivo utiliza punto y coma (`;`)
como delimitador. La carga inicial interpretó la totalidad de los campos como una
única columna, por lo que se realiza una nueva lectura conservando la fuente
original sin modificaciones.

Posteriormente se estandarizan las llaves de identificación de afiliados y personas
a cargo para garantizar compatibilidad con el identificador previamente normalizado
de `T_DIGITURNOS`.

In [59]:
# ============================================================
# 4.3 CORRECCIÓN Y ESTANDARIZACIÓN DE FUENTES DE AFILIACIÓN
# ============================================================

# Relectura correcta de BD_AFILIADOS
afiliados = pd.read_csv(
    "../data/raw/BD_AFILIADOS.csv",
    sep=";",
    low_memory=False
)

print("BD_AFILIADOS REPROCESADA")
print("=" * 70)
print(f"Registros: {len(afiliados):,}")
print(f"Columnas:  {afiliados.shape[1]}")

print("\nColumnas detectadas:")
for i, col in enumerate(afiliados.columns, 1):
    print(f"{i:02d}. {repr(col)}")

BD_AFILIADOS REPROCESADA
Registros: 448,951
Columnas:  10

Columnas detectadas:
01. 'Unnamed: 0'
02. 'id_persona_Modificado'
03. 'NumIdPersona_Modificado'
04. 'Nombre'
05. 'Categoria'
06. 'Segmento_poblacional'
07. 'Piramide1'
08. 'Piramide2'
09. 'RazonSocial'
10. 'Genero'


In [60]:
# ============================================================
# 4.4 ESTANDARIZACIÓN DE LLAVES DE IDENTIFICACIÓN
# ============================================================

# Afiliados
afiliados["id_afiliado"] = pd.to_numeric(
    afiliados["NumIdPersona_Modificado"],
    errors="coerce"
).astype("Int64")

# Grupo familiar
grupo_familiar["id_afiliado_titular"] = pd.to_numeric(
    grupo_familiar["NUM_IDENTIFICACION_AFILIADO"],
    errors="coerce"
).astype("Int64")

grupo_familiar["id_persona_cargo"] = pd.to_numeric(
    grupo_familiar["NUM_IDENTIFICACION_PERSONA_A_CARGO"],
    errors="coerce"
).astype("Int64")

# Control
print("ESTANDARIZACIÓN DE LLAVES")
print("=" * 70)

print(f"Afiliados - registros:                {len(afiliados):,}")
print(f"Afiliados - ID nulos:                 {afiliados['id_afiliado'].isna().sum():,}")
print(f"Afiliados - ID distintos:             {afiliados['id_afiliado'].nunique():,}")

print()
print(f"Grupo familiar - registros:           {len(grupo_familiar):,}")
print(f"ID titular nulos:                     {grupo_familiar['id_afiliado_titular'].isna().sum():,}")
print(f"ID persona a cargo nulos:             {grupo_familiar['id_persona_cargo'].isna().sum():,}")
print(f"Titulares distintos:                  {grupo_familiar['id_afiliado_titular'].nunique():,}")
print(f"Personas a cargo distintas:           {grupo_familiar['id_persona_cargo'].nunique():,}")

print()
print(f"Digiturnos - clientes distintos:      {digiturnos['id_cliente'].nunique():,}")

ESTANDARIZACIÓN DE LLAVES
Afiliados - registros:                448,951
Afiliados - ID nulos:                 10
Afiliados - ID distintos:             303,946

Grupo familiar - registros:           39,733
ID titular nulos:                     0
ID persona a cargo nulos:             1
Titulares distintos:                  29,879
Personas a cargo distintas:           24,554

Digiturnos - clientes distintos:      60,335


In [61]:
# ============================================================
# 4.5 CONTROL DE UNICIDAD DE LLAVES
# ============================================================

duplicados_afiliados = (
    afiliados["id_afiliado"]
    .dropna()
    .duplicated(keep=False)
    .sum()
)

duplicados_personas_cargo = (
    grupo_familiar["id_persona_cargo"]
    .dropna()
    .duplicated(keep=False)
    .sum()
)

print("CONTROL DE UNICIDAD DE LLAVES")
print("=" * 70)

print(f"Filas con ID afiliado duplicado:       {duplicados_afiliados:,}")
print(f"Filas con persona a cargo duplicada:   {duplicados_personas_cargo:,}")

print()
print("Duplicación potencial:")
print(
    "Afiliados:",
    "REVISAR" if duplicados_afiliados > 0 else "Sin duplicados"
)
print(
    "Personas a cargo:",
    "REVISAR" if duplicados_personas_cargo > 0 else "Sin duplicados"
)

CONTROL DE UNICIDAD DE LLAVES
Filas con ID afiliado duplicado:       289,990
Filas con persona a cargo duplicada:   23,034

Duplicación potencial:
Afiliados: REVISAR
Personas a cargo: REVISAR


In [62]:
# ============================================================
# 4.6 CONTROL DE SOLAPAMIENTO ENTRE AFILIADOS Y PERSONAS A CARGO
# ============================================================

ids_afiliados = set(
    afiliados["id_afiliado"]
    .dropna()
    .astype("int64")
)

ids_personas_cargo = set(
    grupo_familiar["id_persona_cargo"]
    .dropna()
    .astype("int64")
)

ids_digiturnos = set(
    digiturnos["id_cliente"]
    .dropna()
    .astype("int64")
)

solapamiento = ids_afiliados.intersection(ids_personas_cargo)

clientes_afiliados = ids_digiturnos.intersection(ids_afiliados)

clientes_grupo_familiar = (
    ids_digiturnos
    .difference(ids_afiliados)
    .intersection(ids_personas_cargo)
)

clientes_no_afiliados = (
    ids_digiturnos
    .difference(ids_afiliados)
    .difference(ids_personas_cargo)
)

print("DIAGNÓSTICO PREVIO DE CLASIFICACIÓN")
print("=" * 70)

print(f"IDs presentes simultáneamente como afiliado y PAC: {len(solapamiento):,}")

print()
print("Clientes únicos de T_DIGITURNOS:")
print(f"Total:                  {len(ids_digiturnos):,}")
print(f"Afiliados directos:     {len(clientes_afiliados):,}")
print(f"Grupo familiar:         {len(clientes_grupo_familiar):,}")
print(f"No afiliados:           {len(clientes_no_afiliados):,}")

print()
print(
    "Reconciliación:",
    len(ids_digiturnos)
    == (
        len(clientes_afiliados)
        + len(clientes_grupo_familiar)
        + len(clientes_no_afiliados)
    )
)

DIAGNÓSTICO PREVIO DE CLASIFICACIÓN
IDs presentes simultáneamente como afiliado y PAC: 10,335

Clientes únicos de T_DIGITURNOS:
Total:                  60,335
Afiliados directos:     47,873
Grupo familiar:         15
No afiliados:           12,447

Reconciliación: True


In [63]:
# ============================================================
# 4.7 CARGA Y VALIDACIÓN DE DIMENSIÓN DE AFILIADOS CONSISTENTES
# ============================================================

ruta_dim_afiliados = DATA_PROCESSED / "dim_afiliados_consistentes.csv"

dim_afiliados = pd.read_csv(
    ruta_dim_afiliados,
    low_memory=False
)

print("DIMENSIÓN DE AFILIADOS CONSISTENTES")
print("=" * 70)
print(f"Registros: {len(dim_afiliados):,}")
print(f"Columnas:  {dim_afiliados.shape[1]}")

print("\nColumnas disponibles:")
for i, col in enumerate(dim_afiliados.columns, 1):
    print(f"{i:02d}. {repr(col)}")

display(dim_afiliados.head())

DIMENSIÓN DE AFILIADOS CONSISTENTES
Registros: 168,866
Columnas:  7

Columnas disponibles:
01. 'id_normalizado'
02. 'Categoria'
03. 'Segmento_poblacional'
04. 'Piramide1'
05. 'Piramide2'
06. 'registros_fuente'
07. 'calidad_perfil'


,id_normalizado,Categoria,Segmento_poblacional,Piramide1,Piramide2,registros_fuente,calidad_perfil
0,10,A,Medio,4 micro,4.5 transaccional,1,Consistente
1,10000000,A,Basico,2 emp medio,2.2 silver,1,Consistente
2,10000004,A,Basico,1 emp grandes,1.1 platinum,2,Consistente
3,10000009,A,Basico,4 micro,4.5 transaccional,2,Consistente
4,10000041,A,Basico,4 micro,4.5 transaccional,2,Consistente


In [64]:
# ============================================================
# 4.8 VALIDACIÓN DE LLAVE EN DIM_AFILIADOS
# ============================================================

print("CONTROL DE LLAVE")
print("=" * 70)

print(
    "Duplicados por identificación:",
    dim_afiliados["id_normalizado"].duplicated().sum()
)

print(
    "Nulos en identificación:",
    dim_afiliados["id_normalizado"].isna().sum()
)

print(
    "Identificaciones distintas:",
    dim_afiliados["id_normalizado"].nunique()
)

CONTROL DE LLAVE
Duplicados por identificación: 0
Nulos en identificación: 0
Identificaciones distintas: 168866


### 4.9 Construcción de la dimensión de grupo familiar

La relación entre afiliados titulares y personas a cargo se transforma en una
dimensión reutilizable con una fila única por persona a cargo.

Dado que una misma persona puede encontrarse asociada a más de un titular, no se
selecciona arbitrariamente un afiliado responsable. La dimensión conserva el número
de titulares asociados y asigna el identificador del titular únicamente cuando la
relación es unívoca.

El resultado se almacena en `data/processed/dim_grupo_familiar.csv` para ser utilizado
de manera consistente por los diferentes ejercicios de la prueba.

In [68]:
# ============================================================
# 4.9 CONSTRUCCIÓN DE DIM_GRUPO_FAMILIAR
# ============================================================

# Copia controlada de la fuente
gf_base = grupo_familiar.copy()

# ------------------------------------------------------------
# 1. Normalización de identificadores
# ------------------------------------------------------------

gf_base["id_persona_cargo"] = pd.to_numeric(
    gf_base["NUM_IDENTIFICACION_PERSONA_A_CARGO"],
    errors="coerce"
).astype("Int64")

gf_base["id_afiliado_titular"] = pd.to_numeric(
    gf_base["NUM_IDENTIFICACION_AFILIADO"],
    errors="coerce"
).astype("Int64")


# ------------------------------------------------------------
# 2. Retirar únicamente registros sin persona a cargo
# ------------------------------------------------------------

registros_fuente_gf = len(gf_base)
nulos_persona_cargo = gf_base["id_persona_cargo"].isna().sum()

gf_base_valida = (
    gf_base[
        gf_base["id_persona_cargo"].notna()
    ]
    [
        ["id_persona_cargo", "id_afiliado_titular"]
    ]
    .drop_duplicates()
    .copy()
)


# ------------------------------------------------------------
# 3. Consolidación a una fila por persona a cargo
# ------------------------------------------------------------

resumen_titulares = (
    gf_base_valida
    .groupby("id_persona_cargo")
    .agg(
        n_titulares=(
            "id_afiliado_titular",
            lambda x: x.dropna().nunique()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Recuperar titular solo cuando es único
# ------------------------------------------------------------

titular_unico = (
    gf_base_valida
    .groupby("id_persona_cargo")["id_afiliado_titular"]
    .agg(
        lambda x:
        x.dropna().iloc[0]
        if x.dropna().nunique() == 1
        else pd.NA
    )
    .reset_index(name="id_afiliado_titular")
)


# ------------------------------------------------------------
# 5. Construcción de la dimensión
# ------------------------------------------------------------

dim_grupo_familiar = resumen_titulares.merge(
    titular_unico,
    on="id_persona_cargo",
    how="left",
    validate="one_to_one"
)

dim_grupo_familiar["estado_relacion"] = np.select(
    [
        dim_grupo_familiar["n_titulares"].eq(1),
        dim_grupo_familiar["n_titulares"].gt(1)
    ],
    [
        "Titular único",
        "Múltiples titulares"
    ],
    default="Sin titular identificable"
)


# ------------------------------------------------------------
# 6. Control
# ------------------------------------------------------------

print("DIMENSIÓN DE GRUPO FAMILIAR")
print("=" * 70)

print(f"Registros fuente:                  {registros_fuente_gf:,}")
print(f"Registros sin persona a cargo:     {nulos_persona_cargo:,}")
print(f"Personas a cargo únicas:           {len(dim_grupo_familiar):,}")
print(
    f"Duplicados en llave final:         "
    f"{dim_grupo_familiar['id_persona_cargo'].duplicated().sum():,}"
)

print()
display(
    dim_grupo_familiar["estado_relacion"]
    .value_counts()
    .rename_axis("estado_relacion")
    .reset_index(name="personas")
)

display(dim_grupo_familiar.head(10))

DIMENSIÓN DE GRUPO FAMILIAR
Registros fuente:                  39,733
Registros sin persona a cargo:     1
Personas a cargo únicas:           24,554
Duplicados en llave final:         0



,estado_relacion,personas
0,Titular único,16698
1,Múltiples titulares,7856


,id_persona_cargo,n_titulares,id_afiliado_titular,estado_relacion
0,149139,1,206257,Titular único
1,157554,1,802720,Titular único
2,172634,1,800493,Titular único
3,196315,1,208593,Titular único
4,227861,1,795722,Titular único
5,245147,1,804186,Titular único
6,250679,1,403825,Titular único
7,251614,1,795204,Titular único
8,251763,1,790598,Titular único
9,251998,1,805057,Titular único


In [69]:
# ============================================================
# 4.10 EXPORTACIÓN DE DIM_GRUPO_FAMILIAR
# ============================================================

ruta_dim_grupo_familiar = (
    DATA_PROCESSED
    / "dim_grupo_familiar.csv"
)

dim_grupo_familiar.to_csv(
    ruta_dim_grupo_familiar,
    index=False,
    encoding="utf-8-sig"
)

print("DIMENSIÓN GENERADA")
print("=" * 70)
print(ruta_dim_grupo_familiar)
print()
print(
    f"Registros exportados: "
    f"{len(dim_grupo_familiar):,}"
)

DIMENSIÓN GENERADA
c:\Users\acoba\Documents\GitHub\Prueba_Analista_BI\data\processed\dim_grupo_familiar.csv

Registros exportados: 24,554


In [70]:
# ============================================================
# 4.11 VALIDACIÓN DE DIMENSIÓN PERSISTIDA
# ============================================================

dim_grupo_familiar = pd.read_csv(
    ruta_dim_grupo_familiar,
    encoding="utf-8-sig",
    low_memory=False
)

dim_grupo_familiar["id_persona_cargo"] = pd.to_numeric(
    dim_grupo_familiar["id_persona_cargo"],
    errors="coerce"
).astype("Int64")

dim_grupo_familiar["id_afiliado_titular"] = pd.to_numeric(
    dim_grupo_familiar["id_afiliado_titular"],
    errors="coerce"
).astype("Int64")

print("VALIDACIÓN DE DIM_GRUPO_FAMILIAR")
print("=" * 70)

print(f"Registros:                {len(dim_grupo_familiar):,}")
print(
    f"IDs únicos:               "
    f"{dim_grupo_familiar['id_persona_cargo'].nunique():,}"
)
print(
    f"Duplicados en llave:      "
    f"{dim_grupo_familiar['id_persona_cargo'].duplicated().sum():,}"
)
print(
    f"Nulos en llave:           "
    f"{dim_grupo_familiar['id_persona_cargo'].isna().sum():,}"
)

VALIDACIÓN DE DIM_GRUPO_FAMILIAR
Registros:                24,554
IDs únicos:               24,554
Duplicados en llave:      0
Nulos en llave:           0


In [72]:
print("CONTROL PRE-MERGE")
print("=" * 70)

print(
    "Duplicados dim_afiliados:",
    dim_afiliados["id_normalizado"].duplicated().sum()
)

print(
    "Duplicados dim_grupo_familiar:",
    dim_grupo_familiar["id_persona_cargo"].duplicated().sum()
)

print(
    "Nulos dim_afiliados:",
    dim_afiliados["id_normalizado"].isna().sum()
)

print(
    "Nulos dim_grupo_familiar:",
    dim_grupo_familiar["id_persona_cargo"].isna().sum()
)

CONTROL PRE-MERGE
Duplicados dim_afiliados: 9
Duplicados dim_grupo_familiar: 0
Nulos dim_afiliados: 10
Nulos dim_grupo_familiar: 0


In [73]:
print("TIPOS DE LLAVE")
print("=" * 70)

print(
    "digiturnos:",
    digiturnos_tiempos["id_cliente"].dtype
)

print(
    "dim_afiliados:",
    dim_afiliados["id_normalizado"].dtype
)

print(
    "dim_grupo_familiar:",
    dim_grupo_familiar["id_persona_cargo"].dtype
)

TIPOS DE LLAVE
digiturnos: Int64
dim_afiliados: Int64
dim_grupo_familiar: Int64


In [74]:
# ============================================================
# 4.12 RECARGA CONTROLADA DE DIMENSIONES MAESTRAS
# ============================================================

ruta_dim_afiliados = DATA_PROCESSED / "dim_afiliados_consistentes.csv"
ruta_dim_grupo_familiar = DATA_PROCESSED / "dim_grupo_familiar.csv"

dim_afiliados = pd.read_csv(
    ruta_dim_afiliados,
    encoding="utf-8-sig",
    low_memory=False
)

dim_grupo_familiar = pd.read_csv(
    ruta_dim_grupo_familiar,
    encoding="utf-8-sig",
    low_memory=False
)

# Estandarización de llaves
dim_afiliados["id_normalizado"] = pd.to_numeric(
    dim_afiliados["id_normalizado"],
    errors="coerce"
).astype("Int64")

dim_grupo_familiar["id_persona_cargo"] = pd.to_numeric(
    dim_grupo_familiar["id_persona_cargo"],
    errors="coerce"
).astype("Int64")

dim_grupo_familiar["id_afiliado_titular"] = pd.to_numeric(
    dim_grupo_familiar["id_afiliado_titular"],
    errors="coerce"
).astype("Int64")

print("VALIDACIÓN DE DIMENSIONES MAESTRAS")
print("=" * 70)

print(
    "dim_afiliados - duplicados:",
    dim_afiliados["id_normalizado"].duplicated().sum()
)
print(
    "dim_afiliados - nulos:",
    dim_afiliados["id_normalizado"].isna().sum()
)

print(
    "dim_grupo_familiar - duplicados:",
    dim_grupo_familiar["id_persona_cargo"].duplicated().sum()
)
print(
    "dim_grupo_familiar - nulos:",
    dim_grupo_familiar["id_persona_cargo"].isna().sum()
)

VALIDACIÓN DE DIMENSIONES MAESTRAS
dim_afiliados - duplicados: 9
dim_afiliados - nulos: 10
dim_grupo_familiar - duplicados: 0
dim_grupo_familiar - nulos: 0
